# Redis Pub/Sub & Redis Streams — Complete Reference

> **Purpose**: A self-contained, interview-ready deep dive into Redis messaging patterns.  
> **Reference**: [hellointerview.com — Redis Deep Dive](https://www.hellointerview.com/learn/system-design/deep-dives/redis)  
> **Last updated**: 2026-04-15

---

## Table of Contents
1. [Why Redis for Messaging?](#1-why-redis-for-messaging)
2. [Redis Pub/Sub](#2-redis-pubsub)
3. [Redis Streams](#3-redis-streams)
4. [Pub/Sub vs Streams — Side-by-Side](#4-pubsub-vs-streams--side-by-side)
5. [When to Use What](#5-when-to-use-what)
6. [Redis vs Kafka vs SQS/SNS](#6-redis-vs-kafka-vs-sqssns)
7. [Key Interview Talking Points](#7-key-interview-talking-points)

---

## 1. Why Redis for Messaging?

Redis is primarily an in-memory data store, but it also supports **two messaging patterns**:

| Pattern | Best For |
|---|---|
| **Pub/Sub** | Fire-and-forget real-time broadcast (chat, live dashboards) |
| **Streams** | Durable, replayable event log (event sourcing, task queues) |

Redis messaging is a **lightweight alternative** to:
- Apache Kafka (high-throughput durable log)
- AWS SQS (managed message queue)
- AWS SNS (managed pub/sub notification service)

The trade-off: Redis is simpler to operate but lacks Kafka's horizontal scale and durability guarantees at extreme volume.

---

## 2. Redis Pub/Sub

### 2.1 How it Works

```
Publisher                 Redis Server                Subscribers
    |                         |                            |
    |  PUBLISH news "hello"   |                            |
    |------------------------>|   push to all subscribers  |
    |                         |--------------------------->|
    |                         |                            |
```

- A **publisher** sends a message to a named **channel**.
- Redis **immediately fans out** that message to every **active subscriber** on that channel.
- No message is stored — it is delivered in real time and then **discarded**.

This is a pure **broadcast** model: one sender, many receivers, zero persistence.

---

### 2.2 Core Commands

| Command | Description |
|---|---|
| `PUBLISH channel message` | Send a message to a channel |
| `SUBSCRIBE ch1 ch2 ...` | Listen to one or more named channels |
| `UNSUBSCRIBE [ch1 ...]` | Stop listening |
| `PSUBSCRIBE pattern` | Subscribe by glob pattern (e.g. `news.*`) |
| `PUNSUBSCRIBE [pattern]` | Unsubscribe from pattern |
| `PUBSUB CHANNELS` | List all active channels |
| `PUBSUB NUMSUB channel` | Count subscribers on a channel |

```bash
# Terminal 1 — Subscriber
SUBSCRIBE news

# Terminal 2 — Publisher
PUBLISH news "Breaking: Redis is fast"
# Terminal 1 receives: "Breaking: Redis is fast"
```

Pattern subscriptions (glob):
```bash
PSUBSCRIBE user:*:notifications   # matches user:42:notifications, user:99:notifications
PSUBSCRIBE order.*                 # matches order.created, order.shipped
```

---

### 2.3 Guarantees & Limitations

| Property | Redis Pub/Sub |
|---|---|
| **Delivery** | At-most-once (fire-and-forget) |
| **Persistence** | None — messages vanish after delivery |
| **Offline subscribers** | Messages dropped — subscriber must be connected |
| **Message history / replay** | Not supported |
| **Acknowledgement** | Not supported |
| **Back-pressure** | Not supported — slow consumer drops messages |
| **Consumer groups** | Not supported |

**Critical mental model**: Pub/Sub is a **live broadcast**, like a radio station. If you're not tuned in at the moment of broadcast, you miss it forever.

---

### 2.4 Sharded Pub/Sub (Redis 7.0+)

In a Redis Cluster, regular Pub/Sub broadcasts to **all nodes** (inefficient). Sharded Pub/Sub routes messages to a **single shard** based on channel hash slot.

```bash
SPUBLISH channel message   # Sharded publish
SSUBSCRIBE channel         # Sharded subscribe
```

Use sharded Pub/Sub in Redis Cluster when you want to reduce cross-node traffic.

---

### 2.5 Real-World Use Cases

- **Live chat** — broadcast messages to all clients in a room
- **Real-time dashboards** — push metric updates to browser clients
- **Cache invalidation** — notify app servers when a cache key changes
- **Presence / online status** — broadcast connect/disconnect events
- **Multiplayer game state** — propagate moves to all players

---

## 3. Redis Streams

### 3.1 What is a Stream?

A Redis Stream is an **append-only log** — a sequence of entries, each with a unique auto-generated ID and a set of key-value fields.

```
Stream: "orders"
─────────────────────────────────────────────────────────────
ID                    Fields
─────────────────────────────────────────────────────────────
1713180000000-0       item=shoes  qty=2  user=alice
1713180001000-0       item=hat    qty=1  user=bob
1713180002000-0       item=bag    qty=3  user=carol
─────────────────────────────────────────────────────────────
```

**ID format**: `<unix-milliseconds>-<sequence>` — always monotonically increasing, so the stream is naturally ordered by time.

Think of it as Redis's version of Kafka's partition log — durable, replayable, supports consumer groups.

---

### 3.2 Core Commands

#### Writing to a Stream

```bash
# XADD stream-name ID field value [field value ...]
# * = auto-generate the ID
XADD orders * item shoes qty 2 user alice
# returns "1713180000000-0"

# Capped stream — keep only last 1000 entries
XADD orders MAXLEN ~ 1000 * item hat qty 1 user bob
```

#### Reading from a Stream

```bash
# XREAD — simple read (no consumer groups)
XREAD COUNT 10 STREAMS orders 0          # read from beginning
XREAD COUNT 10 STREAMS orders 1713180000000-0  # read after a specific ID

# Blocking read — wait up to 5s for new entries
XREAD BLOCK 5000 COUNT 10 STREAMS orders $   # $ = only new entries
```

#### Stream Introspection

```bash
XLEN orders               # number of entries
XRANGE orders - +         # all entries (- = min, + = max)
XRANGE orders - + COUNT 5 # first 5 entries
XREVRANGE orders + -      # reverse order (newest first)
XINFO STREAM orders       # stream metadata
```

---

### 3.3 Consumer Groups — The Key to Reliable Processing

Consumer groups allow **multiple consumers to cooperate** on processing a stream, each getting a **distinct subset** of messages. This enables parallel processing without duplication.

```
Stream: orders
────────────────────────────────────────
  entry-1  entry-2  entry-3  entry-4
────────────────────────────────────────
        ↓ Consumer Group: "processors"
  consumer-A        consumer-B
  [entry-1,3]       [entry-2,4]       ← each entry delivered to exactly one consumer
```

#### Setup

```bash
# Create a consumer group starting from the beginning (0) or only new messages ($)
XGROUP CREATE orders processors $      # only new entries
XGROUP CREATE orders processors 0      # from the very beginning
XGROUP CREATE orders processors 0 MKSTREAM  # create stream if it doesn't exist
```

#### Consuming

```bash
# > = "give me entries not yet delivered to any consumer in this group"
XREADGROUP GROUP processors worker-1 COUNT 5 STREAMS orders >

# Blocking variant
XREADGROUP GROUP processors worker-1 BLOCK 5000 COUNT 5 STREAMS orders >
```

#### Acknowledging

After processing, a consumer **must ACK** the message. Until ACK'd, the entry lives in the **Pending Entry List (PEL)**.

```bash
XACK orders processors 1713180000000-0
```

#### Pending Entry List (PEL) — Crash Recovery

```bash
# See what's pending (unacknowledged) in this group
XPENDING orders processors - + 10

# Claim a stale pending entry (e.g., consumer crashed)
# XCLAIM takes ownership if entry has been idle for > N milliseconds
XCLAIM orders processors worker-2 60000 1713180000000-0

# Auto-claim idle entries (Redis 7.0+)
XAUTOCLAIM orders processors worker-2 60000 0
```

**Recovery flow when a consumer crashes**:
1. Another consumer calls `XPENDING` to find entries that were delivered but never ACK'd
2. Uses `XCLAIM` (or `XAUTOCLAIM`) to take ownership
3. Reprocesses and ACKs

---

### 3.4 Stream Trimming

Streams grow indefinitely by default. Two trim strategies:

```bash
# Exact trim
XADD orders MAXLEN 1000 * ...          # keep exactly last 1000 entries
XTRIM orders MAXLEN 1000               # trim manually

# Approximate trim (faster, uses ~ to allow slight overshoot)
XADD orders MAXLEN ~ 1000 * ...
XTRIM orders MAXLEN ~ 1000

# Time-based trim (Redis 6.2+)
XADD orders MINID ~ 1713100000000 * ... # drop entries older than this ID/timestamp
```

---

### 3.5 Persistence

Redis Streams are stored in Redis memory but benefit from Redis's **persistence mechanisms**:

| Mechanism | Description |
|---|---|
| **RDB snapshots** | Periodic point-in-time dump to disk |
| **AOF (Append-Only File)** | Log every write; replay on restart |
| **RDB + AOF** | Most durable combination |

With AOF enabled, streams survive crashes. Without persistence, streams are lost on restart — just like all Redis data.

---

### 3.6 Real-World Use Cases

- **Event sourcing** — append domain events, replay to rebuild state
- **Activity feeds** — "user X liked post Y" log
- **IoT sensor ingestion** — time-series data from devices
- **Audit logs** — immutable record of system actions
- **Task queues with retry** — use PEL for crash-safe job processing
- **Microservice event bus** — services publish events, other services consume via groups

---

## 4. Pub/Sub vs Streams — Side-by-Side

| Property | Pub/Sub | Streams |
|---|---|---|
| **Delivery guarantee** | At-most-once | At-least-once (with consumer groups + ACK) |
| **Message persistence** | No — ephemeral | Yes — stored in memory, optionally on disk |
| **Replay / history** | No | Yes — read from any past ID |
| **Offline consumers** | Messages lost | Messages wait in stream |
| **Consumer groups** | No | Yes — parallel processing, each msg once |
| **Acknowledgement** | No | Yes — XACK |
| **Crash recovery** | No | Yes — via PEL + XCLAIM |
| **Back-pressure** | No | No (but stream size is bounded via MAXLEN) |
| **Multiple subscribers** | Yes — all receive every message | Yes — multiple groups each get all messages; within a group, each message goes to one consumer |
| **Ordering** | Per-channel, insertion order | Strict, by ID (time-based) |
| **Complexity** | Very simple | Moderate |

### Fan-out behavior difference

```
Pub/Sub:                           Streams (2 consumer groups):
─────────────────────────          ──────────────────────────────────
                                   
Publisher                          Producer
    │                                  │
    ▼                                  ▼
[channel]                         [stream]
    │                              │        │
    ├──► Subscriber A         [group-1]  [group-2]   ← each group gets ALL entries
    ├──► Subscriber B              │          │
    └──► Subscriber C          worker-1   worker-3   ← within a group, load-balanced
                               worker-2   worker-4
```

---

## 5. When to Use What

### Use Redis Pub/Sub when:
- You need **real-time push** with no persistence requirement
- Messages are only useful **right now** (e.g., live cursor position, typing indicator)
- Losing a message is acceptable
- Simplicity is a priority
- All subscribers are always online

### Use Redis Streams when:
- You need **at-least-once delivery**
- Consumers can be **offline** and catch up later
- You need **parallel processing** with exactly-one semantics per group
- You need **replay** (e.g., reprocess events after a bug fix)
- You need **crash recovery** for workers
- Building a **task queue** or **event log**

### Use Kafka instead when:
- You need **multi-broker durability** and replication at scale
- Retention measured in **days/weeks**
- **Very high throughput** (millions of events/sec)
- Cross-datacenter replication
- Ecosystem integrations (Kafka Connect, Kafka Streams)

### Use SQS/SNS instead when:
- You want **fully managed** with zero ops
- AWS ecosystem integration
- SQS: point-to-point queue; SNS: fan-out to many endpoints (Lambda, SQS, HTTP)

---

## 6. Redis vs Kafka vs SQS/SNS

| Dimension | Redis Pub/Sub | Redis Streams | Kafka | SQS | SNS |
|---|---|---|---|---|---|
| **Persistence** | No | Yes (in-memory + disk) | Yes (disk, replicated) | Yes (managed) | No (triggers only) |
| **Durability** | None | Good (with AOF) | Excellent | Excellent | N/A |
| **Throughput** | Very high | High | Very high | High | High |
| **Replay** | No | Yes | Yes | No | No |
| **Consumer groups** | No | Yes | Yes | No (visibility timeout) | No |
| **Ordering** | Per channel | Per stream | Per partition | Best-effort (FIFO queues: per group) | No guarantee |
| **Operational complexity** | Low | Low | High | None (managed) | None (managed) |
| **Best for** | Live broadcast | Reliable event log | High-scale event streaming | Simple work queues | Fan-out notifications |

---

## 7. Key Interview Talking Points

### On Redis Pub/Sub
- "Pub/Sub is at-most-once — no persistence, no replay. If the subscriber is offline, the message is gone."
- "It's ideal for notifications where freshness matters more than completeness — live dashboards, presence, cache invalidation."
- "In Redis Cluster, use Sharded Pub/Sub to avoid broadcasting across all nodes."

### On Redis Streams
- "Streams are an append-only log — think lightweight Kafka inside Redis."
- "Consumer groups give you at-least-once delivery with parallel processing: each entry goes to exactly one consumer within the group."
- "The Pending Entry List (PEL) is the crash-safety mechanism — unACK'd messages stay there so another worker can reclaim them with XCLAIM."
- "Streams are bounded by memory. Use MAXLEN trimming or MINID time-based eviction to manage size."

### Choosing between them
- "If I need guaranteed delivery or the consumer might restart, I'd use Streams. If I just need to push live updates to whoever is connected right now, Pub/Sub is simpler."
- "Streams are a good middle ground between Redis simplicity and Kafka durability for moderate scale."

### Delivery semantics summary
```
At-most-once  →  Pub/Sub         (fire and forget)
At-least-once →  Streams + ACK   (reprocess on crash)
Exactly-once  →  not natively;   need idempotent consumers + deduplication logic
```